# Diabetes Prediction: Data Preprocessing

This notebook prepares the patient-level dataset for machine learning.

The workflow includes:

- loading data from PostgreSQL
- checking data types and missing values
- removing duplicate records
- defining predictors and the diabetes target
- performing a stratified train-test split
- saving the resulting datasets for modelling

Preprocessing transformations such as imputation, scaling, and one-hot encoding will be fitted using the training data only in the modelling notebooks to prevent data leakage.

In [42]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.model_selection import train_test_split

In [43]:
project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.database import get_engine

In [44]:
engine = get_engine()

query = """
SELECT *
FROM ml_dataset;
"""

df = pd.read_sql(query, engine)

engine.dispose()

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 5,764
Columns: 7


In [45]:
df.info()

df.columns.tolist()

<class 'pandas.DataFrame'>
RangeIndex: 5764 entries, 0 to 5763
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               5764 non-null   str    
 1   age              5764 non-null   float64
 2   gender           5764 non-null   str    
 3   bmi              5608 non-null   float64
 4   encounter_count  5764 non-null   int64  
 5   diabetes         5764 non-null   int64  
 6   hypertension     5764 non-null   int64  
dtypes: float64(2), int64(3), str(2)
memory usage: 315.3 KB


['id', 'age', 'gender', 'bmi', 'encounter_count', 'diabetes', 'hypertension']

In [46]:
df["gender"] = df["gender"].astype("category")
df["age"] = df["age"].astype("int64")

In [47]:
duplicate_count = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_count:,}")

Duplicate rows: 0


In [48]:
if "id" in df.columns:
    duplicated_patients = df["id"].duplicated().sum()
    print(f"Duplicated patient IDs: {duplicated_patients:,}")

Duplicated patient IDs: 0


In [49]:
missing_summary = (
    df.isna()
    .sum()
    .to_frame("missing_count")
    .assign(
        missing_percentage=lambda x:
        (x["missing_count"] / len(df) * 100).round(2)
    )
    .sort_values("missing_percentage", ascending=False)
)

missing_summary

,missing_count,missing_percentage
bmi,156,2.71
age,0,0.00
id,0,0.00
gender,0,0.00
encounter_count,0,0.00
diabetes,0,0.00
hypertension,0,0.00


BMI has 156 missing observations (2.71%). Missing BMI values will be imputed using the median of the training dataset within the machine learning pipeline to avoid data leakage.

In [50]:
target_column = "diabetes"

df[target_column].value_counts(dropna=False)

diabetes
0    3324
1    2440
Name: count, dtype: int64

In [51]:
target_distribution = (
    df[target_column]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
    .rename("percentage")
)

target_distribution

diabetes
0    57.67
1    42.33
Name: percentage, dtype: float64

In [52]:
print(df[target_column].dtype)
print(df[target_column].unique())

int64
[1 0]


In [53]:
feature_columns = ["age", "gender", "bmi", "encounter_count", "hypertension"]

In [54]:
X = df[feature_columns].copy()
y = df[target_column].copy()

print(f"Predictor shape: {X.shape}")
print(f"Target shape: {y.shape}")

Predictor shape: (5764, 5)
Target shape: (5764,)


In [55]:
X.head()

,age,gender,bmi,encounter_count,hypertension
0,69,M,28.3,45,0
1,51,F,30.1,81,0
2,25,M,41.7,30,0
3,71,F,30.2,36,0
4,56,F,30.0,28,0


In [59]:
X.info()
y.info()

<class 'pandas.DataFrame'>
RangeIndex: 5764 entries, 0 to 5763
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   age              5764 non-null   int64   
 1   gender           5764 non-null   category
 2   bmi              5608 non-null   float64 
 3   encounter_count  5764 non-null   int64   
 4   hypertension     5764 non-null   int64   
dtypes: category(1), float64(1), int64(3)
memory usage: 186.0 KB
<class 'pandas.Series'>
RangeIndex: 5764 entries, 0 to 5763
Series name: diabetes
Non-Null Count  Dtype
--------------  -----
5764 non-null   int64
dtypes: int64(1)
memory usage: 45.2 KB


In [60]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=123
)

In [61]:
print(X_train.shape)
print(X_test.shape)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(4034, 5)
(1730, 5)
diabetes
0    0.576599
1    0.423401
Name: proportion, dtype: float64
diabetes
0    0.576879
1    0.423121
Name: proportion, dtype: float64


In [62]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

In [64]:
print(X_train.info())
print()
print(X_test.info())
print()
print(y_train.value_counts())
print()
print(y_test.value_counts())

<class 'pandas.DataFrame'>
Index: 4034 entries, 1485 to 3426
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   age              4034 non-null   int64   
 1   gender           4034 non-null   category
 2   bmi              3924 non-null   float64 
 3   encounter_count  4034 non-null   int64   
 4   hypertension     4034 non-null   int64   
dtypes: category(1), float64(1), int64(3)
memory usage: 161.6 KB
None

<class 'pandas.DataFrame'>
Index: 1730 entries, 530 to 3377
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   age              1730 non-null   int64   
 1   gender           1730 non-null   category
 2   bmi              1684 non-null   float64 
 3   encounter_count  1730 non-null   int64   
 4   hypertension     1730 non-null   int64   
dtypes: category(1), float64(1), int64(3)
memory usage: 69.4 KB
None

diabetes
0    23

In [63]:
X_train.to_csv(processed_dir / "X_train.csv", index=False)
X_test.to_csv(processed_dir / "X_test.csv", index=False)

y_train.to_csv(processed_dir / "y_train.csv", index=False)
y_test.to_csv(processed_dir / "y_test.csv", index=False)